# 06 - Hybrid RAG Experiments

Implements Hybrid Retrieval combining dense (vector similarity) and sparse (BM25) retrieval
using Reciprocal Rank Fusion (RRF). This approach aims to capture both semantic and lexical relevance.

**Pipeline:** Query → Dense Retriever + BM25 Retriever → RRF Fusion → LLM Generation → Answer  
**Key Parameter:** `alpha` controls dense vs sparse weighting (0.5 = equal)

In [ ]:
import sys
sys.path.append("..")

from langchain_huggingface import HuggingFaceEmbeddings
from src.data_loader import load_pubmedqa, build_mesh_lookup
from src.chunking import create_documents
from src.ingestion import ingest_to_chroma
from src.retrieval import get_cosine_retriever, BM25Retriever, HybridRetriever
from src.sampling import get_stratified_sample
from src.llm_wrapper import get_groq_llm
from src.rag_pipeline import build_naive_rag_chain, run_rag
import config

## Setup

In [ ]:
df = load_pubmedqa()
mesh_lookup = build_mesh_lookup(df)

embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
llm = get_groq_llm()

documents, ids = create_documents(str(config.DATA_RAW_DIR), mesh_lookup)
stratified_df = get_stratified_sample(df)

## Build Hybrid Retriever

Combines a dense retriever (ChromaDB cosine similarity) with a sparse retriever (BM25)
using Reciprocal Rank Fusion with configurable alpha weighting.

In [ ]:
# Dense retriever
vector_store = ingest_to_chroma(
    documents, ids, embeddings,
    db_name=f"{embedding_key}_pubmed_chroma"
)
dense_retriever = get_cosine_retriever(vector_store)

# Sparse retriever
bm25_retriever = BM25Retriever(documents, k=config.TOP_K)

# Hybrid
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    alpha=config.HYBRID_ALPHA,
)

print(f"Dense retriever: ChromaDB (cosine, k={config.TOP_K})")
print(f"Sparse retriever: BM25 (corpus={len(documents)} docs)")
print(f"Fusion: RRF with alpha={config.HYBRID_ALPHA}")

## Test Hybrid Retrieval

In [ ]:
test_query = stratified_df.iloc[0]["question"]
print(f"Query: {test_query}\n")

dense_results = dense_retriever.invoke(test_query)
sparse_results = bm25_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

print("Dense results:")
for i, doc in enumerate(dense_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

print("\nBM25 results:")
for i, doc in enumerate(sparse_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

print("\nHybrid (RRF) results:")
for i, doc in enumerate(hybrid_results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")

## Run Hybrid RAG

In [ ]:
rag_chain = build_naive_rag_chain(llm)
eval_dataset = run_rag(rag_chain, hybrid_retriever, stratified_df, delay=20)
eval_dataset.to_csv(str(config.RESULTS_RAGAS_DIR / f"hybrid_rag_{embedding_key}_chroma_rrf.csv"))
print(f"Generated {len(eval_dataset)} answers")

## Alpha Sensitivity Analysis

Test different alpha values to understand the dense vs sparse tradeoff.

In [ ]:
# Uncomment to run alpha sweep (resource-intensive)
# for alpha in [0.3, 0.5, 0.7]:
#     hybrid = HybridRetriever(dense_retriever, bm25_retriever, alpha=alpha)
#     results = run_rag(rag_chain, hybrid, stratified_df.head(20), delay=20)
#     results.to_csv(str(config.RESULTS_RAGAS_DIR / f"hybrid_alpha_{alpha}.csv"))
#     print(f"Alpha={alpha}: {len(results)} answers generated")